# H-1B LCA Disclosure Data Analysis — FY2026 Q1
Source: `LCA_Disclosure_Data_FY2026_Q1.xlsx`

In [2]:
import pandas as pd

FILE = "LCA_Disclosure_Data_FY2026_Q1.xlsx"

df = pd.read_excel(FILE, engine="openpyxl")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head(3)

Rows: 83,120  |  Columns: 98


,CASE_NUMBER,CASE_STATUS,RECEIVED_DATE,DECISION_DATE,ORIGINAL_CERT_DATE,VISA_CLASS,JOB_TITLE,SOC_CODE,SOC_TITLE,FULL_TIME_POSITION,...,WILLFUL_VIOLATOR,SUPPORT_H1B,STATUTORY_BASIS,APPENDIX_A_ATTACHED,PUBLIC_DISCLOSURE,PREPARER_LAST_NAME,PREPARER_FIRST_NAME,PREPARER_MIDDLE_INITIAL,PREPARER_BUSINESS_NAME,PREPARER_EMAIL
0,I-200-25356-504116,Certified - Withdrawn,2025-12-22,2025-12-31,2025-12-30,H-1B,Instructor,25-1199.00,"Postsecondary Teachers, All Other",N,...,No,NaN,NaN,NaN,Disclose Business,Silzer,Scot,A,SilzerLaw Chartered,slc@silzerlaw.com
1,I-200-25355-500286,Certified - Withdrawn,2025-12-20,2025-12-31,2025-12-29,H-1B,AI/ML ENGINEER,15-1252.00,Software Developers,Y,...,No,NaN,NaN,NaN,Disclose Business,NaN,NaN,NaN,NaN,NaN
2,I-200-25351-491663,Certified - Withdrawn,2025-12-17,2025-12-31,2025-12-24,H-1B,Research Scientist,19-1042.00,"Medical Scientists, Except Epidemiologists",Y,...,No,NaN,NaN,NaN,Disclose Business,NaN,NaN,NaN,NaN,NaN


## Cluster by Employer Name
Normalise company names (strip whitespace, uppercase) then aggregate petition counts.

In [2]:
# Normalise employer names
df["EMPLOYER_NAME_NORM"] = (
    df["EMPLOYER_NAME"]
    .fillna("UNKNOWN")
    .str.strip()
    .str.upper()
)

# Count petitions (rows) per normalised company name
company_counts = (
    df.groupby("EMPLOYER_NAME_NORM")
    .agg(
        petition_count=("CASE_NUMBER", "count"),
        total_workers=("TOTAL_WORKER_POSITIONS", "sum"),
        certified_count=("CASE_STATUS", lambda s: (s.str.startswith("Certified")).sum()),
    )
    .reset_index()
    .sort_values("petition_count", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique companies: {len(company_counts):,}")
company_counts.head(10)

Unique companies: 17,986


,EMPLOYER_NAME_NORM,petition_count,total_workers,certified_count
0,AMAZON.COM SERVICES LLC,2310,10616,2310
1,APPLE INC.,1638,1638,1637
2,ERNST & YOUNG U.S. LLP,1622,1622,1599
3,COGNIZANT TECHNOLOGY SOLUTIONS US CORP,1385,1385,1385
4,MICROSOFT CORPORATION,1338,1338,1338
5,"META PLATFORMS, INC",858,1098,857
6,DELOITTE CONSULTING LLP,818,1688,817
7,GOOGLE LLC,806,806,803
8,TATA CONSULTANCY SERVICES LIMITED,785,785,780
9,INFOSYS LIMITED,741,2281,736


## Top 20 Companies by Petition Count

In [3]:
top20 = company_counts.head(20).copy()
top20.index = range(1, 21)
top20.columns = ["Company", "Petitions", "Total Workers", "Certified"]
top20

,Company,Petitions,Total Workers,Certified
1,AMAZON.COM SERVICES LLC,2310,10616,2310
2,APPLE INC.,1638,1638,1637
3,ERNST & YOUNG U.S. LLP,1622,1622,1599
4,COGNIZANT TECHNOLOGY SOLUTIONS US CORP,1385,1385,1385
5,MICROSOFT CORPORATION,1338,1338,1338
6,"META PLATFORMS, INC",858,1098,857
7,DELOITTE CONSULTING LLP,818,1688,817
8,GOOGLE LLC,806,806,803
9,TATA CONSULTANCY SERVICES LIMITED,785,785,780
10,INFOSYS LIMITED,741,2281,736


## Bar Chart — Top 20 Most Frequent Employers

In [1]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    top20["Company"].str[:40],   # truncate long names
    top20["Petitions"],
    color="steelblue",
    edgecolor="white",
)

ax.invert_yaxis()   # highest count at top
ax.set_xlabel("Number of LCA Petitions", fontsize=12)
ax.set_title("Top 20 Employers — H-1B LCA Filings FY2026 Q1", fontsize=14, fontweight="bold")
ax.bar_label(bars, fmt="{:,.0f}", padding=4, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("top20_employers.png", dpi=150)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## Summary Stats

In [5]:
total_petitions = len(df)
top20_share = top20["Petitions"].sum() / total_petitions * 100

print(f"Total petitions       : {total_petitions:>10,}")
print(f"Unique employers      : {len(company_counts):>10,}")
print(f"Top-20 share of total : {top20_share:>9.1f}%")

Total petitions       :     83,120
Unique employers      :     17,986
Top-20 share of total :      20.5%
